# Flow matching for simple labelled distributions

In [ ]:
%load_ext autoreload
%autoreload 2

import data  # Auxiliary module for generating toy datasets
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Load toy labelled data

In [ ]:
import numpy as np

target_data, target_labels = data.generate_toy_data("two_gaussians_supervised", n=100000)
n = len(target_data)
source_data = np.random.randn(n, 2)

plotting.plot_distributions(source_data, target_data, target_labels=target_labels)

## Basic flow matching

Let's sample independent couplings between source and target data points

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=n, target_labels=target_labels)

Visualize the generated couplings

In [ ]:
plotting.plot_distributions(source_data, target_data, target_labels=target_labels, couplings=couplings)

Trains a simple MLP as the velocity field estimation network.

In [ ]:
network = models.train_flow_model(couplings, verbose=True)

Visualize the velocity fields we have learned: unconditional and conditional for both classes.

In [ ]:
plotting.plot_class_conditional_velocity_fields(network, source_data, target_data, target_labels=target_labels)

Now that we have the velocity field, we can use a integration method such as Euler's method to run across the flow.

In [ ]:
trajectories = models.compute_trajectories(network, source_data, label=None)
plotting.animate_trajectories(trajectories, target_data=target_data)

We can also generate trajectories conditioned on the class labels to lead the generation process towards a particular class

In [ ]:
data_per_label = {label: source_data[target_labels == label] for label in np.unique(target_labels)}
class_trajectories = {
    label: models.compute_trajectories(network, data_per_label[label], label=label)
    for label in data_per_label
}
plotting.animate_trajectories(class_trajectories, target_data=target_data)

## Rectified flows

By using the generated trajectories to learn a new flow, we can improve the "straightness" of the trajectories, allowing for faster generation.

This also applies to conditioned flows, though we must take care to preserve the labels of the original couplings.

In [ ]:
synthetic_data = np.array([traj[-1][1] for class_trajs in class_trajectories.values() for traj in class_trajs])
synthetic_labels = np.array([label for label, class_trajs in class_trajectories.items() for _ in class_trajs])

# Shuffle the data and labels together
indices = np.arange(len(synthetic_data))
np.random.shuffle(indices)
synthetic_data = synthetic_data[indices]
synthetic_labels = synthetic_labels[indices]

couplings = [(src, tgt, label) for src, tgt, label in zip(source_data, synthetic_data, synthetic_labels)]
plotting.plot_distributions(source_data, synthetic_data, couplings=couplings, target_labels=synthetic_labels)

In [ ]:
rectified_network = models.train_flow_model(couplings, verbose=True)

Let's check now the rectified velocity fields

In [ ]:
plotting.plot_class_conditional_velocity_fields(rectified_network, source_data, target_data, target_labels=target_labels)

The trajectories should be straigther in the new fields

In [ ]:
rectified_trajectories = models.compute_trajectories(rectified_network, source_data, label=None)
plotting.animate_trajectories(rectified_trajectories, target_data=target_data)

In [ ]:
data_per_label = {label: source_data[target_labels == label] for label in np.unique(target_labels)}
rectified_class_trajectories = {
    label: models.compute_trajectories(rectified_network, data_per_label[label], label=label)
    for label in data_per_label
}
plotting.animate_trajectories(rectified_class_trajectories, target_data=target_data)

Finally, we can compare the data generated by the rectified flow against the original data from both classes.

In [ ]:
plotting.plot_class_conditioned_generated_data_comparison(target_data, rectified_class_trajectories, target_labels)